## 환경설정

In [0]:
%sh
# 필수 Azure SDK 패키지 설치
uv pip install azure-identity azure-keyvault-secrets azure-storage-file-datalake

In [0]:
import sys
import os

os.environ["KEY_VAULT_URL"] = "https://kv-3dt-team1.vault.azure.net/"

sys.path.insert(0, "/Workspace/Repos/3dt030@msacademy.msai.kr/3dt-2nd-project/src")

# ✅ import 전에 싱글톤 초기화
import utils.vault_manager
utils.vault_manager._instance = None

# ✅ 그 다음 vault 새로 가져오기
from utils.vault_manager import get_vault_manager
vault = get_vault_manager()

print("client_id:", vault.get_secret("adls-client-id"))
print("client_secret:", vault.get_secret("adls-client-secret"))
print("tenant_id:", vault.get_secret("adls-tenant-id"))

## 데이터 로드

In [0]:
vault.get_storage_client("3dtteam1adls")

from pyspark.sql import functions as F
from pyspark.sql.types import StringType

BRONZE_BASE = "abfss://raw@3dtteam1adls.dfs.core.windows.net/news/naver/"

# 와일드카드로 JSON 파일 직접 읽기
df_raw = (spark.read
    .option("mergeSchema", "true")
    .option("multiLine", "true")   # ← 이거 추가!
    .json(f"{BRONZE_BASE}source=*/month=*/*.json")
)

print(f"총 로드: {df_raw.count():,}건")
print(f"컬럼: {df_raw.columns}")
df_raw.printSchema()


In [0]:
# 경로 자동 생성 (파일명 신경 안 써도 됨)
sources = ["samsung", "skhynix"]
months = (
    [f"2025-{m:02d}" for m in range(4, 13)] +  # 2025-04 ~ 2025-12
    [f"2026-{m:02d}" for m in range(1, 5)]      # 2026-01 ~ 2026-04
)

paths = [
    f"{BRONZE_BASE}source={s}/month={m}/"
    for s in sources
    for m in months
]

print(f"총 경로 수: {len(paths)}개")

df_raw = (spark.read
    .option("mergeSchema", "true")
    .option("multiLine", "true")
    .json(paths)
)

print(f"총 로드: {df_raw.count():,}건")
print(f"컬럼: {df_raw.columns}")


### 컬럼 표준화 + pubDate 처리

In [0]:
df = (df_raw
    .withColumnRenamed("newsId",       "news_id")
    .withColumnRenamed("pubDate",      "pub_date_raw")
    .withColumnRenamed("adjustedDate", "adjusted_date")
    .withColumnRenamed("fetchedAt",    "fetched_at")
    .withColumnRenamed("source",       "stock_keyword")  # samsung/skhynix
)

# pubDate 전체 null → adjustedDate 대체
df = df.withColumn(
    "pub_date",
    F.coalesce(
        F.to_date("pub_date_raw",  "yyyy-MM-dd"),
        F.to_date("adjusted_date", "yyyy-MM-dd")
    )
)

# news_source는 항상 naver
df = df.withColumn("news_source", F.lit("naver"))

print(f"컬럼: {df.columns}")
display(df.limit(10))

In [0]:
df = df.drop("naverUrl", "pub_date_raw", "adjusted_date")

df = df.withColumnRenamed("fetched_at", "published_time")

print(f"최종 컬럼: {df.columns}")
display(df.limit(10))

In [0]:
# 컬럼 순서 정리
df = df.select(
    "news_id",           # PK
    "stock_keyword",     # samsung/skhynix
    "news_source",       # naver
    "pub_date",          # 발행 날짜
    "published_time",    # 발행 시각
    "press",             # 언론사
    "headline",          # 기사 제목
    "body",              # 원문 본문
    "description",       # 3줄 요약 (LLM 예정)
    "url",               # 원문 URL
)

print(f"컬럼 순서: {df.columns}")
display(df.limit(10))


### samsung, skhynix 중복 기사 전처리
- url 기준으로 중복 처리 
- 겹치는 기사를 stock_keyword->samsung,skhynix 설정 후 중복된 기사 제거

In [0]:
# 1. 한글 → 영어 표준화
df = df.withColumn(
    "stock_keyword",
    F.when(F.col("stock_keyword") == "삼성전자", "samsung")
     .when(F.col("stock_keyword") == "SK하이닉스", "skhynix")
     .otherwise(F.col("stock_keyword"))
)

# 2. URL 기준으로 머지
# - news_id는 버리고 (samsung용/skhynix용 각각 다르게 생성됐으므로)
# - URL이 같으면 stock_keyword만 합치고 나머지는 첫 번째 값 사용
df_merged = df.groupBy("url").agg(
    F.first("pub_date").alias("pub_date"),
    F.first("published_time").alias("published_time"),
    F.first("news_source").alias("news_source"),
    F.first("press").alias("press"),
    F.first("headline").alias("headline"),
    F.first("body").alias("body"),
    F.first("description").alias("description"),
    F.concat_ws(",", F.collect_set("stock_keyword")).alias("stock_keyword")
)

# 3. 결과 확인
print(f"머지 전: {df.count():,}건")
print(f"머지 후: {df_merged.count():,}건")
print(f"중복 제거: {df.count() - df_merged.count():,}건")

display(
    df_merged.groupBy("stock_keyword")
             .count()
             .orderBy("count", ascending=False)
)


### 본문 html, 특수문자 등 정리

In [0]:
from pyspark.sql import functions as F
import re

# 1. HTML 및 불필요한 텍스트 제거 UDF
def clean_text(text):
    if text is None:
        return None
    # HTML 태그 제거
    text = re.sub(r'<[^>]+>', '', text)
    # HTML 엔티티 제거 (&amp; &lt; 등)
    text = re.sub(r'&[a-zA-Z]+;', '', text)
    # 특수문자 정리 (문장부호 제외)
    text = re.sub(r'[\r\n\t]+', ' ', text)
    # 연속 공백 정리
    text = re.sub(r' +', ' ', text)
    return text.strip()

clean_udf = F.udf(clean_text)

# 2. body, headline, description 컬럼 클렌징
df_cleaned = df_merged.withColumn("body", clean_udf(F.col("body"))) \
                      .withColumn("headline", clean_udf(F.col("headline"))) \
                      .withColumn("description", clean_udf(F.col("description")))

# 3. 키워드별 10개씩 샘플 출력
keywords = ["samsung", "skhynix", "skhynix,samsung"]

for kw in keywords:
    print(f"\n{'='*60}")
    print(f"📰 stock_keyword = '{kw}' 샘플 10건")
    print(f"{'='*60}")
    df_cleaned.filter(F.col("stock_keyword") == kw) \
              .select("stock_keyword", "headline", "body") \
              .limit(10) \
              .show(truncate=100)

### description 전처리
- **Azure OpenAI GPT-4o mini** 를 사용하여 본문(`body`)을 3줄 요약
- `description`이 없는 기사만 요약 처리 (있는 건 그대로 유지)
- **Batch API** 방식으로 제출 (최대 24시간 대기, 비용 50% 절감)
- 50,000건씩 분할하여 `.jsonl` 파일로 제출
- 1회차 전체 처리 후 이후 증분 데이터는 실시간 API로 처리

#### 셀 1 - Batch 요청 파일 생성

In [0]:
import json
from pyspark.sql import functions as F

# description 없는 것만 추출
df_need_summary = df_cleaned.filter(
    F.col("description").isNull() | (F.col("description") == "")
).select("news_id", "body").toPandas()

print(f"[INFO] 요약 필요 건수: {len(df_need_summary)}건")

# Batch API 요청 파일 생성 (50,000건씩 분할)
batch_size = 50000
chunks = [df_need_summary[i:i+batch_size] for i in range(0, len(df_need_summary), batch_size)]

for idx, chunk in enumerate(chunks):
    output_file = f"/tmp/batch_request_{idx+1}.jsonl"
    with open(output_file, "w", encoding="utf-8") as f:
        for _, row in chunk.iterrows():
            request = {
                "custom_id": str(row["news_id"]),
                "method": "POST",
                "url": "/chat/completions",
                "body": {
                    "model": "gpt-4o-mini",
                    "messages": [
                        {
                            "role": "system",
                            "content": "뉴스 기사를 3줄로 요약해주세요. 핵심 내용만 간결하게 작성하세요."
                        },
                        {
                            "role": "user",
                            "content": str(row["body"])[:3000]  # 토큰 절약
                        }
                    ],
                    "max_tokens": 300
                }
            }
            f.write(json.dumps(request, ensure_ascii=False) + "\n")
    print(f"[OK] 파일 생성: {output_file} ({len(chunk)}건)")

#### 셀 2 - Batch 파일 업로드 및 작업 제출

In [0]:
from openai import AzureOpenAI

# Azure OpenAI 클라이언트
client = AzureOpenAI(
    api_key=vault.get_secret("azure-openai-key"),      # Key Vault에서 가져오기
    api_version="2024-07-01-preview",
    azure_endpoint=vault.get_secret("azure-openai-endpoint")
)

batch_job_ids = []

for idx in range(len(chunks)):
    file_path = f"/tmp/batch_request_{idx+1}.jsonl"
    
    # 파일 업로드
    with open(file_path, "rb") as f:
        uploaded_file = client.files.create(file=f, purpose="batch")
    print(f"[OK] 파일 업로드: {uploaded_file.id}")
    
    # Batch 작업 제출
    batch_job = client.batches.create(
        input_file_id=uploaded_file.id,
        endpoint="/chat/completions",
        completion_window="24h"
    )
    batch_job_ids.append(batch_job.id)
    print(f"[OK] Batch 제출: {batch_job.id}")

print(f"\n[INFO] 전체 Batch Job IDs: {batch_job_ids}")

#### 셀 3 - 상태 확인 (나중에 실행)

In [0]:
# 제출 후 나중에 상태 확인
for job_id in batch_job_ids:
    job = client.batches.retrieve(job_id)
    print(f"Job {job_id}: {job.status} | 완료: {job.request_counts.completed} / 전체: {job.request_counts.total}")

#### 셀 4 - 결과 수집 및 df_final 생성

In [0]:
import pandas as pd

all_summaries = {}

for job_id in batch_job_ids:
    job = client.batches.retrieve(job_id)
    
    if job.status != "completed":
        print(f"[WARN] {job_id} 아직 미완료: {job.status}")
        continue
    
    # 결과 다운로드
    result_content = client.files.content(job.output_file_id).text
    
    for line in result_content.strip().split("\n"):
        result = json.loads(line)
        news_id = result["custom_id"]
        summary = result["response"]["body"]["choices"][0]["message"]["content"]
        all_summaries[news_id] = summary

print(f"[OK] 요약 수집 완료: {len(all_summaries)}건")

# Spark DataFrame으로 변환
summary_df = spark.createDataFrame(
    pd.DataFrame(list(all_summaries.items()), columns=["news_id", "description_new"])
)

# 기존 df_cleaned와 합치기
df_has_summary = df_cleaned.filter(
    F.col("description").isNotNull() & (F.col("description") != "")
)
df_need_summary_spark = df_cleaned.filter(
    F.col("description").isNull() | (F.col("description") == "")
).drop("description").join(summary_df, on="news_id", how="left") \
 .withColumnRenamed("description_new", "description")

df_final = df_has_summary.union(df_need_summary_spark)
print(f"[OK] 최종 데이터: {df_final.count()}건")

## ------------------------------------------------------------------------------
#### 하루치 테스트 코드

In [0]:
from pyspark.sql import functions as F

test_date = "2025-04-01"

df_test = df_cleaned.filter(F.col("pub_date") == test_date)
df_test_need = df_test.filter(
    F.col("description").isNull() | (F.col("description") == "")
)

print(f"[INFO] 테스트 데이터: {df_test.count()}건 ({test_date})")
print(f"[INFO] 요약 필요: {df_test_need.count()}건")

# 컬럼 확인
print(f"[INFO] 컬럼 목록: {df_test_need.columns}")

In [0]:
# 셀 2 - Batch 요청 파일 생성
import json

df_test_pd = df_test_need.select("url", "body").toPandas()

output_file = "/tmp/batch_test.jsonl"
with open(output_file, "w", encoding="utf-8") as f:
    for _, row in df_test_pd.iterrows():
        request = {
            "custom_id": str(row["url"]),
            "method": "POST",
            "url": "/chat/completions",
            "body": {
                "model": "gpt-4o-mini",
                "messages": [
                    {
                        "role": "system",
                        "content": "당신은 뉴스 요약 전문가입니다. 주어진 뉴스 기사를 핵심 내용 위주로 3줄로 요약해주세요."
                    },
                    {
                        "role": "user",
                        "content": str(row["body"])[:3000]
                    }
                ],
                "max_tokens": 300
            }
        }
        f.write(json.dumps(request, ensure_ascii=False) + "\n")

print(f"[OK] 파일 생성 완료: {df_test_pd.shape[0]}건")

In [0]:
# 셀 3 - Batch 제출

from openai import AzureOpenAI

client = AzureOpenAI(
    api_key=vault.get_secret("azure-openai-key"),
    api_version="2024-07-01-preview",
    azure_endpoint=vault.get_secret("azure-openai-endpoint")
)

with open("/tmp/batch_test.jsonl", "rb") as f:
    uploaded_file = client.files.create(file=f, purpose="batch")
print(f"[OK] 파일 업로드: {uploaded_file.id}")

batch_job = client.batches.create(
    input_file_id=uploaded_file.id,
    endpoint="/chat/completions",
    completion_window="24h"
)
batch_job_id = batch_job.id
print(f"[OK] Batch 제출: {batch_job_id}")
print(f"[INFO] 상태: {batch_job.status}")

In [0]:
# 셀 4 - 상태 확인 (나중에 실행)
job = client.batches.retrieve(batch_job_id)
print(f"상태: {job.status}")
print(f"완료: {job.request_counts.completed} / 전체: {job.request_counts.total}")

In [0]:
# 셀 5 - 결과 수집 및 확인
import pandas as pd

# 완료 확인
job = client.batches.retrieve(batch_job_id)
if job.status != "completed":
    print(f"[WARN] 아직 미완료: {job.status}")
else:
    result_content = client.files.content(job.output_file_id).text
    summaries = {}
    
    for line in result_content.strip().split("\n"):
        result = json.loads(line)
        news_id = result["custom_id"]
        summary = result["response"]["body"]["choices"][0]["message"]["content"]
        summaries[news_id] = summary
    
    print(f"[OK] 요약 수집: {len(summaries)}건")
    
    # 샘플 출력
    for news_id, summary in list(summaries.items())[:3]:
        print(f"\n{'='*60}")
        print(f"news_id: {news_id}")
        print(f"요약:\n{summary}")